In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
scores=pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
thresholds=pd.read_csv('FINDER/Official/daily_thresholds.csv')
frames=[]
for market,code in [('^GSPC','SP500'),('FTSE100','FTSE'),('GDAXI','DAX'),('CACT','CAC'),('STOXX','STOXX'),('CBU0.L','CBU0')]:
    frame=scores[scores.market.eq(market)].copy()
    frame['score']=frame.decision_supervisor
    supervisor=thresholds[thresholds.market.eq(market)&thresholds.agent.eq('decision_supervisor')][['target_date','threshold','prediction']]
    frame=frame.merge(supervisor.rename(columns={'threshold':'supervisor_threshold','prediction':'supervisor_prediction'}),on='target_date',validate='one_to_one')
    for model,key in [('IND','lstm_ind'),('NEWS','lstm_news')]:
        folder=Path('Benchmark/Official')/f'{code}_{model}'
        benchmark=pd.read_csv(folder/'test_predictions.csv')
        aligned=frame[['target_date','actual_label']].merge(benchmark,on='target_date',validate='one_to_one')
        assert len(aligned)==len(frame) and aligned.actual_label.eq(aligned.actual).all()
        frame=frame.merge(benchmark[['target_date','probability','prediction']].rename(columns={'probability':key,'prediction':key+'_prediction'}),on='target_date',validate='one_to_one')
        frame[key+'_threshold']=json.loads((folder/'metrics.json').read_text())['threshold']
    frames.append(frame)
joined=pd.concat(frames,ignore_index=True)
expected=pd.read_csv('FigureSources/daily_plot_data.csv')
joined=joined[expected.columns].sort_values(['market','target_date']).reset_index(drop=True)
expected=expected.sort_values(['market','target_date']).reset_index(drop=True)
pd.testing.assert_frame_equal(joined,expected,check_dtype=False,check_exact=False,rtol=1e-12,atol=1e-14)
output=Path('Figures/Generated')
output.mkdir(parents=True,exist_ok=True)
joined.to_csv(output/'daily_plot_data.csv',index=False)


In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MaxNLocator
plt.rcdefaults()
out = Path('Figures/Generated')
out.mkdir(parents=True, exist_ok=True)
metrics = pd.read_csv('FINDER/Official/reported_comparison.csv')
daily = pd.read_csv('FINDER/Official/daily_scores_all_markets.csv')
markets = [('^GSPC', 'SP500', 'S&P 500'), ('FTSE100', 'FTSE', 'FTSE 100'), ('GDAXI', 'DAX', 'DAX'), ('CACT', 'CAC', 'CACT'), ('STOXX', 'STOXX', 'STOXX'), ('CBU0.L', 'CBU0', 'CBU0.L')]
series_names = ['Decision Supervisor', 'LSTM-IND', 'LSTM-(IND+NEWS)']
series_colors = ['royalblue', '#555F6B', '#555F6B']
prevalence = daily.groupby('market').actual_label.mean()
plt.rcdefaults()
plt.rcParams['pdf.fonttype'] = 42
checks = []
for metric in ['auc', 'ap']:
    plt.figure(figsize=(5.3, 3.2))
    for j, (name, color, marker) in enumerate(zip(series_names, series_colors, ['o', 's', '^'])):
        values = metrics[metrics.model.eq(name)].set_index('market').loc[[m[2] for m in markets], metric]
        plt.scatter(values, np.arange(6) + (j - 1) * 0.035, color=color, marker=marker, s=50, edgecolor='white', linewidth=0.65, zorder=3 + j)
    if metric == 'auc':
        plt.axvline(0.5, color='black', linestyle='-', linewidth=0.8)
        plt.xlim(0, 1)
        plt.xticks(np.arange(0, 1.01, 0.2), fontsize=12)
    else:
        plt.xlim(0, 0.26)
        plt.xticks(np.arange(0, 0.26, 0.05), fontsize=12)
        plt.scatter([prevalence[m[0]] for m in markets], np.arange(6), marker='|', s=110, color='black', linewidth=0.8, zorder=2)
    plt.yticks(np.arange(6), [m[2] for m in markets], fontsize=12)
    plt.ylim(5.45, -0.45)
    plt.xlabel('ROC-AUC' if metric == 'auc' else 'Average precision', fontsize=13)
    for y in np.arange(0.5, 5, 1):
        plt.axhline(y, linestyle='--', color='black', alpha=0.2, linewidth=0.7, zorder=0)
    plt.tight_layout()
    fig, ax = (plt.gcf(), plt.gca())
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    labels = [ax.xaxis.label, *ax.get_xticklabels(), *ax.get_yticklabels()]
    clipped = [label.get_text() for label in labels if not fig.bbox.contains(*label.get_window_extent(renderer).get_points()[0]) or not fig.bbox.contains(*label.get_window_extent(renderer).get_points()[1])]
    assert not clipped, clipped
    assert len(ax.get_yticks()) == 6 and all((spine.get_visible() for spine in ax.spines.values()))
    assert [collection.get_offsets().shape[0] for collection in ax.collections[:3]] == [6, 6, 6]
    for collection, name in zip(ax.collections[:3], series_names):
        expected = metrics[metrics.model.eq(name)].set_index('market').loc[[m[2] for m in markets], metric].to_numpy()
        assert np.array_equal(collection.get_offsets()[:, 0], expected)
    checks.append({'metric': metric, 'dimensions_inches': [5.3, 3.2], 'font': plt.rcParams['font.sans-serif'][0], 'visible_spines': list(ax.spines), 'model_y_offsets': [-0.035, 0, 0.035], 'horizontal_separator_alpha': 0.2, 'xlim': list(ax.get_xlim()), 'market_spacing_points': (ax.transData.transform((0, 1))[1] - ax.transData.transform((0, 0))[1]) * 72 / fig.dpi, 'references': 0.5 if metric == 'auc' else {m[2]: float(prevalence[m[0]]) for m in markets}, 'clipped_labels': clipped, 'plotted_values_equal_formal_metrics': True})
    plt.savefig(out / f'market_{metric}_compact.png', dpi=360, facecolor='white')
    plt.close()
markets = [('^GSPC', 'SP500', 'S&P 500'), ('FTSE100', 'FTSE', 'FTSE 100'), ('GDAXI', 'DAX', 'DAX'), ('CACT', 'CAC', 'CACT'), ('STOXX', 'STOXX', 'STOXX'), ('CBU0.L', 'CBU0', 'CBU0.L')]
keys = ['decision_supervisor', 'lstm_ind', 'lstm_news']
labels = ['Decision Supervisor', 'LSTM-IND', 'LSTM-(IND+NEWS)']
colors = ['#4169E1', '#555F6B', '#555F6B']
threshold_keys = ['supervisor_threshold', 'lstm_ind_threshold', 'lstm_news_threshold']
prediction_keys = ['supervisor_prediction', 'lstm_ind_prediction', 'lstm_news_prediction']
stats, checks, marker_checks = ([], [], [])
source = Path('FigureSources/daily_plot_data.csv')
joined = pd.read_csv(source, parse_dates=['target_date', 'information_cutoff'])
reference_stats = pd.read_csv(Path('FigureSources/standardization_parameters.csv')).set_index(['market', 'series'])
assert len(joined) == 2279 and np.allclose(joined.score, joined.decision_supervisor)
for market, code, name in markets:
    frame = joined[joined.market.eq(market)]
    for key in keys:
        saved = reference_stats.loc[market, key]
        assert np.isclose(frame[key].mean(), saved['mean']) and np.isclose(frame[key].std(ddof=0), saved['std_ddof0'])
for market, code, name in markets:
    full = joined[joined.market.eq(market)].sort_values('target_date').copy()
    for scope in ['daily', 'april'] if market == '^GSPC' else ['daily']:
        frame = full if scope == 'daily' else full[full.target_date.between('2025-04-01', '2025-04-14')].copy()
        for scale in ['raw'] if scope == 'april' else ['standardized']:
            is_case = scope == 'april'
            fig, axes = plt.subplots(4, 1, figsize=(5.5, 4.1) if is_case else (6.8, 2.1), sharex=True, gridspec_kw={'height_ratios': [2, 2, 2, 1]})
            fig.subplots_adjust(left=0.12 if is_case else 0.08, right=0.985 if is_case else 0.995, bottom=0.1 if is_case else 0.13, top=0.92, hspace=0.1)
            dates = frame.target_date
            x = np.arange(len(frame)) if is_case else mdates.date2num(dates)
            title = 'S&P 500 · 1–14 April 2025' if is_case else name
            fig.text(0.12 if is_case else 0.08, 0.973, title, weight='bold', fontsize=7.5 if is_case else 7.1, va='top')
            standardized_bounds = np.concatenate([(full[k].to_numpy() - full[k].mean()) / full[k].std(ddof=0) for k in keys] + [(full[t].to_numpy() - full[k].mean()) / full[k].std(ddof=0) for k, t in zip(keys, threshold_keys)])
            common_low, common_high = (np.floor(standardized_bounds.min() - 0.25), np.ceil(standardized_bounds.max() + 1.15))
            for i, (key, t, label, color, prediction) in enumerate(zip(keys, threshold_keys, labels, colors, prediction_keys)):
                values, cutoff = (frame[key].to_numpy(), frame[t].to_numpy())
                mu, sd = (full[key].mean(), full[key].std(ddof=0))
                stats.append({'market': market, 'scope': scope, 'scale': scale, 'series': key, 'mean': mu, 'std_ddof0': sd, 'reference_start': str(full.target_date.min().date()), 'reference_end': str(full.target_date.max().date())})
                if scale == 'standardized':
                    values, cutoff = ((values - mu) / sd, (cutoff - mu) / sd)
                ax = axes[i]
                ax.plot(x, values, color=color, lw=1.1, zorder=4)
                ax.step(x, cutoff, where='mid' if is_case else 'post', color='#111111', ls=(0, (4, 2)), lw=0.8, zorder=3)
                actual = frame.actual_label.astype(bool).to_numpy()
                predicted = frame[prediction].astype(bool).to_numpy()
                true_positive = actual & predicted
                false_negative = actual & ~predicted
                false_positive = ~actual & predicted
                ax.scatter(x[true_positive], values[true_positive], s=16 if is_case else 6, marker='o', c='#B43C4B', linewidths=0, zorder=8)
                ax.scatter(x[false_negative], values[false_negative], s=20 if is_case else 8, marker='X', c='#B43C4B', edgecolors='white', linewidths=0.12, zorder=9)
                ax.scatter(x[false_positive], values[false_positive], s=20 if is_case else 8, marker='^', c='#AD7625', linewidths=0, zorder=8)
                if is_case:
                    true_negative = ~actual & ~predicted
                    ax.scatter(x[true_negative], values[true_negative], s=9, marker='o', c=color, linewidths=0, zorder=8)
                if scale == 'raw':
                    ax.set_ylim(-0.04, 1.18)
                    ax.set_yticks([0, 0.5, 1])
                else:
                    ax.set_ylim(common_low, common_high)
                    ax.yaxis.set_major_locator(MaxNLocator(4, prune='both'))
                    ax.axhline(0, color='#b5bcc5', lw=0.4, zorder=1)
                ax.text(0.008, 0.95, label, transform=ax.transAxes, fontsize=6.5 if is_case else 6.2, weight='bold', va='top', bbox={'facecolor': 'white', 'edgecolor': 'none', 'pad': 1.3, 'alpha': 0.9}, zorder=6)
                ax.set_ylabel(('Score' if scale == 'raw' else 'z-score') if is_case or i == 1 else '', labelpad=6, fontsize=7 if is_case else 6.6)
            ax = axes[3]
            barcolors = np.where(frame.actual_label.eq(1), '#B43C4B', '#89929D')
            ax.bar(x, frame.actual_log_return * 100, width=0.4 if is_case else 1.0, color=barcolors, linewidth=0, zorder=3)
            ax.plot(x, frame.crash_threshold * 100, color='#111111', ls=(0, (4, 2)), lw=0.8, zorder=4)
            ax.axhline(0, color='#828b95', lw=0.5, zorder=2)
            ax.set_ylabel('Return (%)', labelpad=6, fontsize=7 if is_case else 6.6)
            ax.yaxis.set_major_locator(MaxNLocator(3))
            for a in axes:
                a.set_facecolor('none')
                a.grid(False)
                a.tick_params(axis='x', length=0, labelbottom=False)
                a.tick_params(axis='y', length=2, pad=3, labelsize=6.5 if is_case else 6.2)
            axes[-1].tick_params(axis='x', labelbottom=True, pad=5, labelsize=6.5 if is_case else 6.2)
            if is_case:
                ax.set_xticks(x, dates.dt.strftime('%d %b'))
                ax.set_xlim(-0.4, len(frame) - 0.6)
            else:
                month_ticks = pd.date_range(dates.min().replace(day=1), dates.max(), freq='MS')
                month_ticks = month_ticks[month_ticks >= dates.min()]
                ax.set_xticks(mdates.date2num(month_ticks), [d.strftime('%y-%m') if d.month == 1 else d.strftime('%m') for d in month_ticks])
                ax.set_xlim(x[0] - 4, x[-1] + 4)
            fig.canvas.draw()
            renderer = fig.canvas.get_renderer()
            for axis in axes:
                assert all((spine.get_visible() for spine in axis.spines.values()))
                visible_y = [tick for value, tick in zip(axis.get_yticks(), axis.get_yticklabels()) if min(axis.get_ylim()) <= value <= max(axis.get_ylim())]
                for text in [axis.yaxis.label, *visible_y, *axis.get_xticklabels()]:
                    if text.get_visible():
                        box = text.get_window_extent(renderer)
                        assert fig.bbox.contains(*box.get_points()[0]) and fig.bbox.contains(*box.get_points()[1]), (scope, code, scale, text.get_text(), box.bounds)
            label_boxes = [axis.yaxis.label.get_window_extent(renderer) for axis in axes if axis.yaxis.label.get_text()]
            assert not any((a.overlaps(b) for a, b in zip(label_boxes, label_boxes[1:]))), (scope, code, scale, 'y-axis label overlap')
            stem = f'april_{code}_{scale}_visible_outcomes' if is_case else f'daily_{code}_{scale}_visible_outcomes'
            fig.savefig(out / (stem + '.png'), dpi=360, facecolor='white')
            checks.append({'figure': stem, 'observations': len(frame), 'events': int(frame.actual_label.sum()), 'panel_ratio': [2, 2, 2, 1], 'outcome_markers': {'TP': 'red circle', 'FN': 'filled red cross', 'FP': 'filled amber triangle', 'TN': 'series-colored dot' if is_case else 'line only'}, 'visible_spines': 4, 'scale': scale, 'clipped_labels': 0})
            if scope == 'april' and scale == 'raw':
                plt.show()
            else:
                plt.close(fig)
data = joined
series = [('score', 'Supervisor', '#4169E1', '-'), ('lstm_ind', 'LSTM-IND', '#555F6B', '-'), ('lstm_news', 'LSTM-(IND+NEWS)', '#555F6B', (0, (3, 2)))]
fig, axes = plt.subplots(1, 6, figsize=(8.4, 2.0), sharey=True)
fig.subplots_adjust(left=0.06, right=0.985, bottom=0.24, top=0.88, wspace=0.12)
stats = []
for ax, (market, title) in zip(axes, [(m[0], m[2]) for m in markets]):
    frame = data[data.market.eq(market)].sort_values('target_date')
    for key, label, color, style in series:
        ax.plot(frame.target_date, frame[key], color=color, ls=style, lw=0.65 if key == 'score' else 0.9, alpha=0.9 if key == 'score' else 1, zorder=2 if key == 'score' else 3)
        stats.append({'market': market, 'series': key, 'observations': len(frame), 'mean': frame[key].mean(), 'std_ddof0': frame[key].std(ddof=0), 'minimum': frame[key].min(), 'maximum': frame[key].max()})
    ax.scatter(frame.loc[frame.actual_label.eq(1), 'target_date'], [1.015] * int(frame.actual_label.sum()), marker='o', s=6, edgecolors='white', linewidths=0.15, color='#B43C4B', alpha=1, clip_on=False)
    ax.set_title(title, fontsize=8, weight='bold', pad=7)
    ax.set_ylim(0, 1.04)
    ax.set_xlim(frame.target_date.min(), frame.target_date.max())
    ax.set_box_aspect(1)
    ax.set_yticks([0, 0.5, 1], ['0', '0.5', '1'])
    ax.set_xticks(mdates.date2num(pd.to_datetime(['2025-04-01', '2026-01-01', '2026-08-01'])), ['Apr\n2025', 'Jan\n2026', 'Aug\n2026'])
    ax.tick_params(axis='both', labelsize=7, length=2, pad=3)
    ax.get_xticklabels()[0].set_ha('left')
    ax.get_xticklabels()[-1].set_ha('right')
axes[0].set_ylabel('Score', fontsize=7.5, labelpad=4)
fig.text(0.52, 0.045, 'Target month', ha='center', fontsize=7, color='black')
fig.canvas.draw()
boxes = [ax.get_window_extent() for ax in axes]
assert all((abs(box.width / box.height - 1) < 1e-09 for box in boxes))
assert len(data) == 2279 and len(axes) == 6
renderer = fig.canvas.get_renderer()
for axis in axes:
    for text in [axis.title, axis.yaxis.label, *axis.get_xticklabels(), *axis.get_yticklabels()]:
        if text.get_visible():
            box = text.get_window_extent(renderer)
            assert fig.bbox.contains(*box.get_points()[0]) and fig.bbox.contains(*box.get_points()[1]), text.get_text()
assert not any((left.get_xticklabels()[-1].get_window_extent(renderer).overlaps(right.get_xticklabels()[0].get_window_extent(renderer)) for left, right in zip(axes, axes[1:])))
fig.savefig(out / 'raw_scores_six_markets_outcomes.png', dpi=360, facecolor='white')
checks.append({'figure': 'raw_scores_six_markets_outcomes', 'layout': [1, 6], 'legend': False, 'panel_aspect_ratios': [box.width / box.height for box in boxes], 'observations': len(data), 'visible_spines': 4})
plt.show()
for source_name, target_name in {'market_auc_compact.png': 'Figure 08 A ROC AUC.png', 'market_ap_compact.png': 'Figure 08 B Average Precision.png', 'raw_scores_six_markets_outcomes.png': 'Figure 09 Risk Scores.png', 'april_SP500_raw_visible_outcomes.png': 'Figure 11 April Case Study.png', 'daily_SP500_standardized_visible_outcomes.png': 'Figure 10 A SP500.png', 'daily_FTSE_standardized_visible_outcomes.png': 'Figure 10 B FTSE100.png', 'daily_DAX_standardized_visible_outcomes.png': 'Figure 10 C DAX.png', 'daily_CAC_standardized_visible_outcomes.png': 'Figure 10 D CACT.png', 'daily_STOXX_standardized_visible_outcomes.png': 'Figure 10 E STOXX.png', 'daily_CBU0_standardized_visible_outcomes.png': 'Figure 10 F CBU0.png'}.items():
    (out / source_name).replace(out / target_name)
